In [ ]:
import time
import yaml
import torch
import torch.nn as nn
import numpy as np
from pathlib import Path
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from torchmetrics.classification import BinaryF1Score, MulticlassAccuracy
from torcheval.metrics import Throughput
from argparse import ArgumentParser
import logging
from datetime import datetime

from qcm.hybrid_models.hybrid_reupload import HybridReuploadClassifier 
from qcm.hybrid_models.hybrid_reupload import QuantumHeadReupload
from qcm.data.datasets import get_dataloaders

from train_reupload import train_epoch

In [11]:
def load_config(path: str) -> dict:
    with open(path, 'r') as f:
        return yaml.safe_load(f)
config = load_config('../configs/config_reupload.yaml')
dataset_type = config.get('dataset_type', 'pcam')

In [12]:
model = HybridReuploadClassifier(config=config, use_quantum=True)

In [13]:
train_loader, val_loader = get_dataloaders(config)

In [14]:
# Setup optimizer, scheduler, and loss function
optimizer = Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=config['training']['lr'])
scheduler = CosineAnnealingLR(optimizer, T_max=config['training']['epochs'])

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = HybridReuploadClassifier(config=config, use_quantum=True)
model.to(device)

HybridReuploadClassifier(
  (backbone): PCAMBackbone(
    (features): Sequential(
      (0): Conv2d(3, 2, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (1): BatchNorm2d(2, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): Conv2d(2, 4, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (4): BatchNorm2d(4, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): ReLU()
      (6): Conv2d(4, 8, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (7): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (8): ReLU()
      (9): AdaptiveAvgPool2d(output_size=(2, 2))
    )
    (latent): Sequential(
      (0): Linear(in_features=32, out_features=9, bias=True)
      (1): Tanh()
    )
  )
  (head): QuantumHeadReupload()
  (normalisation): Tanh()
)

In [16]:
# Define run name and log file path
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_name = f"{dataset_type}_quantum_{timestamp}"
log_filepath = Path("./logs/training_log.csv")
log_filepath.parent.mkdir(exist_ok=True)

In [17]:
writer = SummaryWriter(log_dir=f"runs/test")
train_epoch_losses = []
val_epoch_losses = []

In [18]:
for epoch in range(config['training']['epochs']):
    avg_train_loss = train_epoch(model, train_loader, optimizer, scheduler, epoch, config, writer)
    # avg_val_loss = validate(model, val_loader, epoch, config, writer)
    
    train_epoch_losses.append(avg_train_loss)
    # val_epoch_losses.append(avg_val_loss)

2025-10-03 13:46:35,490 - INFO - Starting training epoch 0
2025-10-03 13:46:37,367 - INFO - Finished training epoch 0. Avg Loss: 0.2699
2025-10-03 13:46:37,369 - INFO - Starting training epoch 1
2025-10-03 13:46:39,144 - INFO - Finished training epoch 1. Avg Loss: 0.2711
2025-10-03 13:46:39,146 - INFO - Starting training epoch 2
2025-10-03 13:46:40,918 - INFO - Finished training epoch 2. Avg Loss: 0.2812
2025-10-03 13:46:40,920 - INFO - Starting training epoch 3
2025-10-03 13:46:42,729 - INFO - Finished training epoch 3. Avg Loss: 0.2745
2025-10-03 13:46:42,731 - INFO - Starting training epoch 4
2025-10-03 13:46:44,497 - INFO - Finished training epoch 4. Avg Loss: 0.2687
2025-10-03 13:46:44,499 - INFO - Starting training epoch 5
2025-10-03 13:46:46,273 - INFO - Finished training epoch 5. Avg Loss: 0.2914
2025-10-03 13:46:46,275 - INFO - Starting training epoch 6
2025-10-03 13:46:48,045 - INFO - Finished training epoch 6. Avg Loss: 0.2685
2025-10-03 13:46:48,047 - INFO - Starting traini

In [20]:
img, label = train_loader.dataset[1]
x = model.backbone(img.reshape(1,3,96,96))

In [21]:
model.head.q_params

Parameter containing:
tensor([[ 1.7532,  0.8089,  0.1300,  1.1947, -0.4961, -0.9164,  0.1764,  0.1964,
         -0.7006]], requires_grad=True)

In [22]:
import h5py

In [2]:
import numpy as np
data = np.load('../data/toy/train_latents_48.npy')

In [3]:
data.shape

(262144, 48)